# Pandas — Part 2 (GroupBy, Merge & Missing Data)

In [ ]:
# Day 25 — Pandas Part 2
# In Day 24, we learned the fundamentals:
#   Series, DataFrames, selecting, filtering, sorting, CSV
#
# Today we go DEEPER into the operations you'll use in real projects:
#   1. groupby() — split data into groups and compute aggregations
#   2. agg() — apply multiple functions at once
#   3. apply() — custom transformations
#   4. merge() / join() — combining two DataFrames
#   5. concat() — stacking DataFrames
#   6. Handling missing data (NaN) — isna(), fillna(), dropna()
#   7. value_counts() — count unique values
#
# These are the operations that make pandas POWERFUL
# You'll use groupby and merge in almost every data analysis task

### groupby() — Split, Apply, Combine

In [ ]:
# groupby() is the pandas version of SQL's GROUP BY
# It splits data into groups, applies a function, then combines results
#
# Real-world analogy:
# You have a box of mixed fruits. You want the total weight PER fruit type.
# Step 1: SPLIT — separate apples, oranges, mangoes into groups
# Step 2: APPLY — weigh each group
# Step 3: COMBINE — put the results together

import pandas as pd

data = {
    'server': ['web-01', 'web-02', 'db-01', 'cache-01', 'web-03', 'db-02', 'cache-02', 'web-04'],
    'cpu':    [45, 87, 92, 55, 78, 88, 42, 63],
    'memory': [62, 78, 85, 40, 71, 90, 38, 55],
    'region': ['eastus', 'eastus', 'westeurope', 'eastus', 'westeurope',
               'westeurope', 'eastus', 'eastus'],
    'team':   ['frontend', 'frontend', 'backend', 'infra', 'backend',
               'backend', 'infra', 'frontend']
}
df = pd.DataFrame(data)
print(df)

In [ ]:
# Average CPU per region
print(df.groupby('region')['cpu'].mean())
# region
# eastus        58.4
# westeurope    86.0

# Read it as: "Group the data by region, then take the cpu column, then calculate mean"

# Count of servers per team
print(df.groupby('team')['server'].count())
# team
# backend     3
# frontend    3
# infra       2

# Max CPU per team
print(df.groupby('team')['cpu'].max())

In [ ]:
# agg() — apply MULTIPLE functions at once

# Multiple stats for CPU grouped by region
result = df.groupby('region')['cpu'].agg(['mean', 'min', 'max', 'count'])
print(result)
#             mean  min  max  count
# region
# eastus      58.4   42   87      5
# westeurope  86.0   78   92      3

# Different functions for different columns
result2 = df.groupby('region').agg({
    'cpu':    ['mean', 'max'],
    'memory': ['mean', 'max'],
    'server': 'count'
})
print(result2)

In [ ]:
# Group by MULTIPLE columns

result3 = df.groupby(['region', 'team'])['cpu'].mean()
print(result3)
# region      team
# eastus      frontend    65.0
#             infra       48.5
# westeurope  backend     86.0

# Reset index to make it a regular DataFrame
result3_df = df.groupby(['region', 'team'])['cpu'].mean().reset_index()
result3_df.columns = ['region', 'team', 'avg_cpu']
print(result3_df)

### apply() — Custom Transformations

In [ ]:
# apply() lets you run ANY function on each value/row/column
# Think of it as a flexible map() for DataFrames

import pandas as pd

data = {
    'server': ['web-01', 'web-02', 'db-01', 'cache-01'],
    'cpu':    [45, 87, 92, 55],
    'memory': [62, 78, 85, 40]
}
df = pd.DataFrame(data)

# apply() on a Series (single column) — works like map
def categorise_cpu(cpu):
    if cpu > 90:
        return 'CRITICAL'
    elif cpu > 70:
        return 'WARNING'
    else:
        return 'OK'

df['cpu_status'] = df['cpu'].apply(categorise_cpu)
print(df)

# Using lambda with apply (shorter for simple operations)
df['cpu_doubled'] = df['cpu'].apply(lambda x: x * 2)
print(df)

In [ ]:
# apply() on entire rows (axis=1)
# axis=0 -> apply to each column (default)
# axis=1 -> apply to each ROW

def health_check(row):
    if row['cpu'] > 90 or row['memory'] > 90:
        return 'CRITICAL'
    elif row['cpu'] > 70 or row['memory'] > 70:
        return 'WARNING'
    return 'HEALTHY'

df['health'] = df.apply(health_check, axis=1)    # axis=1 -> row by row
print(df[['server', 'cpu', 'memory', 'health']])

### value_counts() — Count Unique Values

In [ ]:
# value_counts() is a quick way to count occurrences of each unique value
# Works on a single column (Series)

import pandas as pd

data = {
    'server': ['web-01', 'web-02', 'db-01', 'cache-01', 'web-03', 'db-02', 'cache-02', 'web-04'],
    'status': ['running', 'running', 'running', 'stopped', 'running', 'running', 'stopped', 'running'],
    'region': ['eastus', 'eastus', 'westeurope', 'eastus', 'westeurope',
               'westeurope', 'eastus', 'eastus']
}
df = pd.DataFrame(data)

print(df['status'].value_counts())
# running    6
# stopped    2

print(df['region'].value_counts())
# eastus        5
# westeurope    3

# Get as percentages
print(df['status'].value_counts(normalize=True))
# running    0.75
# stopped    0.25

# Number of unique values
print('Unique regions:', df['region'].nunique())    # 2

### merge() — Combining DataFrames (like SQL JOIN)

In [ ]:
# merge() combines two DataFrames based on a COMMON column
# Like SQL JOIN or Excel VLOOKUP

import pandas as pd

# DataFrame 1: Server metrics
metrics = pd.DataFrame({
    'server': ['web-01', 'web-02', 'db-01', 'cache-01'],
    'cpu':    [45, 87, 92, 55],
    'memory': [62, 78, 85, 40]
})

# DataFrame 2: Server details
details = pd.DataFrame({
    'server': ['web-01', 'web-02', 'db-01', 'cache-01'],
    'ip':     ['10.0.0.1', '10.0.0.2', '10.0.0.3', '10.0.0.4'],
    'team':   ['frontend', 'frontend', 'backend', 'infra']
})

# Merge on the common column 'server'
merged = pd.merge(metrics, details, on='server')
print(merged)
#     server  cpu  memory        ip      team
# 0   web-01   45      62  10.0.0.1  frontend
# 1   web-02   87      78  10.0.0.2  frontend
# 2    db-01   92      85  10.0.0.3   backend
# 3  cache-01  55      40  10.0.0.4     infra

# Both tables combined into ONE based on the 'server' column!

In [ ]:
# Types of merge (join types)

import pandas as pd

left = pd.DataFrame({'key': ['A', 'B', 'C'], 'value1': [1, 2, 3]})
right = pd.DataFrame({'key': ['B', 'C', 'D'], 'value2': [20, 30, 40]})

print('Left:')
print(left)
print('\nRight:')
print(right)

# inner (default) — only rows where key exists in BOTH tables
print('\n--- inner ---')
print(pd.merge(left, right, on='key', how='inner'))    # B, C

# left — all rows from LEFT, matching from right (NaN if no match)
print('\n--- left ---')
print(pd.merge(left, right, on='key', how='left'))     # A, B, C

# right — all rows from RIGHT, matching from left (NaN if no match)
print('\n--- right ---')
print(pd.merge(left, right, on='key', how='right'))    # B, C, D

# outer — all rows from BOTH tables (NaN where no match)
print('\n--- outer ---')
print(pd.merge(left, right, on='key', how='outer'))    # A, B, C, D

### concat() — Stacking DataFrames

In [ ]:
# concat() stacks DataFrames on top of each other (or side by side)
# Unlike merge which JOINS on a key, concat just APPENDS rows

import pandas as pd

# Monthly reports that need to be combined
jan = pd.DataFrame({'server': ['web-01', 'web-02'], 'cpu': [45, 87], 'month': ['Jan', 'Jan']})
feb = pd.DataFrame({'server': ['web-01', 'web-02'], 'cpu': [50, 82], 'month': ['Feb', 'Feb']})
mar = pd.DataFrame({'server': ['web-01', 'web-02'], 'cpu': [48, 79], 'month': ['Mar', 'Mar']})

# Stack vertically (default)
combined = pd.concat([jan, feb, mar], ignore_index=True)
print(combined)
#   server  cpu month
# 0  web-01   45   Jan
# 1  web-02   87   Jan
# 2  web-01   50   Feb
# 3  web-02   82   Feb
# 4  web-01   48   Mar
# 5  web-02   79   Mar

# ignore_index=True resets the index (0, 1, 2, ... instead of repeating 0, 1)

### Handling Missing Data (NaN)

In [ ]:
# In real data, values are often MISSING
# Pandas uses NaN (Not a Number) to represent missing values
# Handling missing data is CRITICAL for clean analysis

import pandas as pd
import numpy as np

data = {
    'server': ['web-01', 'web-02', 'db-01', 'cache-01', 'web-03'],
    'cpu':    [45, np.nan, 92, 55, np.nan],       # 2 missing values
    'memory': [62, 78, np.nan, 40, 71],            # 1 missing value
    'status': ['running', 'running', None, 'stopped', 'running']  # None also = missing
}
df = pd.DataFrame(data)
print(df)
print()

In [ ]:
# Detecting missing values

# isna() — returns True where value is missing
print(df.isna())

# Count missing values per column
print('\nMissing per column:')
print(df.isna().sum())

# Total missing values
print(f'\nTotal missing: {df.isna().sum().sum()}')

# Percentage of missing values
print('\nMissing %:')
print((df.isna().sum() / len(df) * 100).round(1))

In [ ]:
# Handling missing values — 3 strategies

# Strategy 1: DROP rows with missing values
cleaned = df.dropna()     # drops ANY row that has at least 1 NaN
print('After dropna():')
print(cleaned)

# Drop only rows where SPECIFIC columns are NaN
cleaned2 = df.dropna(subset=['cpu'])    # only drop if cpu is missing
print('\nDrop only if cpu missing:')
print(cleaned2)

In [ ]:
# Strategy 2: FILL missing values

import pandas as pd
import numpy as np

data = {
    'server': ['web-01', 'web-02', 'db-01', 'cache-01', 'web-03'],
    'cpu':    [45, np.nan, 92, 55, np.nan],
    'memory': [62, 78, np.nan, 40, 71],
    'status': ['running', 'running', None, 'stopped', 'running']
}
df = pd.DataFrame(data)

# Fill with a specific value
df_filled = df.fillna({'cpu': 0, 'memory': 0, 'status': 'unknown'})
print('Fill with 0/unknown:')
print(df_filled)

# Fill with the MEAN (common for numerical data)
df['cpu'] = df['cpu'].fillna(df['cpu'].mean())
df['memory'] = df['memory'].fillna(df['memory'].mean())
print('\nFill with mean:')
print(df)

# Fill with forward fill (use previous value)
# df['status'] = df['status'].ffill()   # fills with the value above

In [ ]:
# Practice Questions:

# 1. Using the server data, group by 'region' and find:
#    a) Average CPU and memory per region
#    b) Count of servers per region
#    c) The server with max CPU in each region

# 2. Create two DataFrames:
#    students: (student_id, name, class)
#    marks: (student_id, subject, score)
#    Merge them to get: student_id, name, class, subject, score

# 3. Create a DataFrame with some NaN values.
#    a) Count missing values per column
#    b) Fill numeric NaN with column mean
#    c) Fill string NaN with 'unknown'
#    d) Drop rows where MORE THAN 2 columns are NaN

# 4. Given monthly server reports (Jan, Feb, Mar DataFrames),
#    combine them using concat and then group by server
#    to find average CPU across all months

### DevOps / AI-MLOps Practice

In [ ]:
# These are the pandas operations you'll use in real data pipelines

# Exercise 1: Infrastructure Cost Report
# Create a DataFrame of Azure resources:
# resources = pd.DataFrame({
#   'name': ['prod-web-01', 'prod-web-02', 'prod-db-01', 'dev-web-01', 'dev-db-01',
#            'staging-web-01', 'prod-cache-01', 'dev-cache-01'],
#   'type': ['VM', 'VM', 'Database', 'VM', 'Database', 'VM', 'Cache', 'Cache'],
#   'region': ['eastus', 'eastus', 'westeurope', 'eastus', 'eastus',
#              'westeurope', 'eastus', 'eastus'],
#   'env': ['prod', 'prod', 'prod', 'dev', 'dev', 'staging', 'prod', 'dev'],
#   'monthly_cost': [150, 150, 300, 75, 150, 100, 80, 40],
#   'team': ['frontend', 'frontend', 'backend', 'frontend', 'backend',
#            'frontend', 'infra', 'infra']
# })
# a) Total cost per environment (prod/dev/staging)
# b) Total cost per team
# c) Average cost per resource type
# d) Create a pivot: rows=team, columns=env, values=sum of monthly_cost

# Exercise 2: MLOps — Experiment Comparison
# Create two DataFrames:
# experiments: (exp_id, model, dataset, hyperparams)
# results: (exp_id, accuracy, f1, precision, recall, status)
# a) Merge them on exp_id
# b) Filter only 'completed' experiments
# c) Group by model and find mean accuracy
# d) Find the best model-dataset combination
# e) Handle any missing metrics (fill with 0 or drop)

# Exercise 3: Log Analysis Pipeline
# Simulate a log DataFrame:
# logs = pd.DataFrame({
#   'timestamp': pd.date_range('2026-07-01', periods=100, freq='h'),
#   'level': np.random.choice(['INFO', 'WARN', 'ERROR'], 100, p=[0.7, 0.2, 0.1]),
#   'service': np.random.choice(['api', 'web', 'db', 'cache'], 100),
#   'response_time': np.random.normal(200, 50, 100)
# })
# a) Count logs per level (value_counts)
# b) Average response time per service
# c) Find hours with ERROR logs
# d) Create a summary: service × level count table